# QA Chain, Agent, and Conversation Memory

Pipeline: load persisted vectorstore → build RAG chain → wrap in LangChain agent with tools → add conversation memory → test end to end.

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(dotenv_path=Path("../.env"))

VECTORSTORE_DIR = Path("../data/vectorstore")

print("Env loaded.")
print(f"OpenAI key present: {bool(os.getenv('OPENAI_API_KEY'))}")
print(f"LangSmith key present: {bool(os.getenv('LANGCHAIN_API_KEY'))}")

Env loaded.
OpenAI key present: True
LangSmith key present: True


## 1. Load the Persisted Vector Store

Load ChromaDB from disk using the same embedding model used to build it. No re-embedding happens here.

In [2]:
import warnings
warnings.filterwarnings("ignore")

from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma

embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=os.getenv("OPENAI_API_KEY"),
)

vectorstore = Chroma(
    persist_directory=str(VECTORSTORE_DIR),
    embedding_function=embedding_model,
    collection_name="youtube_qa",
)

print(f"Vector store loaded: {vectorstore._collection.count()} documents")

Vector store loaded: 514 documents


## 2. Initialize the LLM

GPT-4o-mini — cost efficient and more than capable for RAG-based QA. Temperature 0 keeps answers factual and consistent.

In [3]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    openai_api_key=os.getenv("OPENAI_API_KEY"),
)

print("LLM ready.")

LLM ready.


## 3. Build the Retrieval Tool

We wrap the retriever as a LangChain tool so the agent can decide when to use it. The description tells the agent what this tool is for.

In [4]:
from langchain_core.tools.retriever import create_retriever_tool

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4},
)

retriever_tool = create_retriever_tool(
    retriever=retriever,
    name="youtube_transcript_search",
    description=(
        "Search across YouTube video transcripts covering education, tech/AI, and entertainment. "
        "Use this tool whenever the user asks a question about any of the video content."
    ),
)

print("Retriever tool ready.")

Retriever tool ready.


## 4. Add a General Knowledge Tool

A second tool lets the agent answer questions that fall outside the video transcripts — keeping the bot useful for broader conversations.

In [5]:
from langchain_core.tools import tool

@tool
def general_knowledge(query: str) -> str:
    """
    Answer general knowledge questions that are not covered by the YouTube transcripts.
    Use this when the user asks something outside the scope of the video content.
    """
    response = llm.invoke(query)
    return response.content

print("General knowledge tool ready.")

General knowledge tool ready.


## 5. Set Up Conversation Memory

ConversationBufferWindowMemory keeps the last 5 exchanges so the agent maintains context across a conversation without inflating the prompt.

In [6]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

message_history = ChatMessageHistory()

def get_session_history(session_id: str):
    return message_history

print("Memory ready.")

Memory ready.


## 6. Build the Agent

We use the OpenAI Tools agent — it lets the LLM decide which tool to call based on the user's question. The system prompt sets the agent's persona and scope.

In [8]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langgraph.prebuilt import create_react_agent

system_prompt = """You are a helpful assistant with access to YouTube video transcripts 
covering education, tech/AI, and entertainment topics. 

When answering questions:
- Always search the transcripts first for relevant information
- Reference the video title and category when citing information from transcripts
- If the question is outside the scope of the transcripts, use your general knowledge
- Keep answers clear and concise
- Maintain context from the conversation history
"""

tools = [retriever_tool, general_knowledge]

agent_executor = create_react_agent(
    model=llm,
    tools=tools,
    prompt=system_prompt,
)

agent_with_memory = RunnableWithMessageHistory(
    agent_executor,
    get_session_history,
    input_messages_key="messages",
    history_messages_key="chat_history",
)

print("Agent ready.")

Agent ready.


## 7. Test the Agent

Run a few questions across different categories to confirm the agent is routing correctly, using memory, and citing sources.

In [9]:
from langchain_core.messages import HumanMessage

test_questions = [
    "How do neural networks learn?",
    "What did Erwin say in his speech?",
    "What was discussed in the GPT-4 announcement?",
    "Can you summarize what we just talked about?",
]

session_id = "test_session"

for question in test_questions:
    print(f"Question: {question}")
    print("-" * 60)
    response = agent_with_memory.invoke(
        {"messages": [HumanMessage(content=question)]},
        config={"configurable": {"session_id": session_id}},
    )
    print(f"Answer: {response['messages'][-1].content}")
    print()

Question: How do neural networks learn?
------------------------------------------------------------
Answer: Neural networks learn through a training process that involves adjusting the weights of connections between neurons based on the data they encounter. Here’s a concise overview of how this works:

1. **Architecture**: A neural network consists of layers of interconnected nodes (neurons), including an input layer, hidden layers, and an output layer.

2. **Forward Propagation**: Input data is fed into the network, where each neuron processes the input using a weighted sum and a non-linear activation function. This continues through the layers until the output layer produces a prediction.

3. **Loss Calculation**: The network's prediction is compared to the actual target value using a loss function, which quantifies the difference between the prediction and the actual value.

4. **Backpropagation**: To improve predictions, the network calculates the gradient of the loss function wit

### Multilingual Handling Test

In [10]:
bilingual_questions = [
    "What topics did Bad Bunny discuss in his Hot Ones interview?",
    "Did Bad Bunny speak Spanish during the interview? If so, what did he say?",
    "How did Bad Bunny react to the hot sauces?",
]

session_id = "bilingual_test"

for question in bilingual_questions:
    print(f"Question: {question}")
    print("-" * 60)
    response = agent_with_memory.invoke(
        {"messages": [HumanMessage(content=question)]},
        config={"configurable": {"session_id": session_id}},
    )
    print(f"Answer: {response['messages'][-1].content}")
    print()

Question: What topics did Bad Bunny discuss in his Hot Ones interview?
------------------------------------------------------------
Answer: In his Hot Ones interview, Bad Bunny discussed several topics, including:

1. **His Music and New Album**: He talked about his latest album, "Deb terar Mas," and shared insights about his favorite song from the album, including the backstory related to salsa lessons and the music video.

2. **Cultural Impact**: Bad Bunny mentioned how his music serves as an invitation for people to learn Spanish, highlighting the cultural significance of his work.

3. **Interviews and Career Evolution**: He reflected on how his relationship with doing interviews in the United States has changed over time, noting that he still struggles with English but is enjoying the process more now.

4. **Personal Experiences**: He shared his experiences with hot sauces, mentioning that he enjoys homemade hot sauces from Puerto Rico.

These topics were discussed while he tackled